# Firecrawl Connector

This notebook shows how to connect Claude to the web using [Firecrawl](https://firecrawl.dev) — a web scraping, search, and interaction API that returns clean markdown optimized for LLM context windows.

Firecrawl provides three integration paths:

1. **REST API** — Call Firecrawl endpoints directly from Python code
2. **Agent Skills & CLI** — Install the Firecrawl CLI and skills for AI coding agents
3. **MCP Server** — Add Firecrawl as a Model Context Protocol server for tool-use workflows

This notebook walks through all three.

## Prerequisites

You will need:
- A [Firecrawl API key](https://www.firecrawl.dev/signin?view=signup) (free tier available)
- An [Anthropic API key](https://console.anthropic.com/) for Claude

---
## Part 1: REST API Integration

Use Firecrawl's REST API to search the web, scrape pages, and feed the results to Claude.

### 1.1 Setup

In [ ]:
%pip install anthropic requests firecrawl-py

In [ ]:
import os

# Set your API keys
FIRECRAWL_API_KEY = os.environ.get("FIRECRAWL_API_KEY", "fc-YOUR_API_KEY")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "sk-YOUR_API_KEY")

### 1.2 Search the Web

Firecrawl's `/search` endpoint finds pages by query and optionally returns full page content as clean markdown.

In [ ]:
import requests

FIRECRAWL_BASE_URL = "https://api.firecrawl.dev/v1"

def firecrawl_search(query: str, limit: int = 5, scrape_options: dict = None) -> dict:
    """Search the web using Firecrawl and return results with optional full page content."""
    headers = {
        "Authorization": f"Bearer {FIRECRAWL_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "query": query,
        "limit": limit,
    }
    if scrape_options:
        payload["scrapeOptions"] = scrape_options

    response = requests.post(
        f"{FIRECRAWL_BASE_URL}/search",
        headers=headers,
        json=payload,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()

In [ ]:
search_results = firecrawl_search("Claude anthropic model context protocol", limit=3)

for i, result in enumerate(search_results.get("data", [])):
    print(f"Result {i+1}: {result.get('title', 'N/A')}")
    print(f"  URL: {result.get('url', 'N/A')}")
    print(f"  Content preview: {result.get('markdown', '')[:150]}...")
    print()

### 1.3 Scrape a Page

When you already have a URL, use `/scrape` to extract clean markdown content.

In [ ]:
def firecrawl_scrape(url: str, formats: list = None) -> dict:
    """Scrape a single URL and return clean markdown content."""
    headers = {
        "Authorization": f"Bearer {FIRECRAWL_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {"url": url}
    if formats:
        payload["formats"] = formats

    response = requests.post(
        f"{FIRECRAWL_BASE_URL}/scrape",
        headers=headers,
        json=payload,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()

In [ ]:
scrape_result = firecrawl_scrape("https://docs.firecrawl.dev")

markdown_content = scrape_result.get("data", {}).get("markdown", "")
print(f"Scraped {len(markdown_content)} characters of markdown")
print(f"Preview:\n{markdown_content[:500]}")

### 1.4 Feed Results to Claude

Now combine Firecrawl's web data with Claude to answer questions using real-time information.

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def ask_claude_with_web_context(question: str, num_results: int = 3) -> str:
    """Search the web with Firecrawl, then ask Claude to answer using the results."""
    # Step 1: Search the web
    search_results = firecrawl_search(question, limit=num_results)

    # Step 2: Format results for Claude
    formatted_results = []
    for i, result in enumerate(search_results.get("data", [])):
        formatted_results.append(
            f'<source index="{i+1}">\n'
            f'<url>{result.get("url", "")}</url>\n'
            f'<title>{result.get("title", "")}</title>\n'
            f'<content>{result.get("markdown", result.get("description", ""))}</content>\n'
            f'</source>'
        )
    context = "\n".join(formatted_results)

    # Step 3: Ask Claude
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": f"""Here are web search results:\n{context}\n\nUsing only the search results above, answer this question concisely. Cite the source URLs.\n\nQuestion: {question}""",
            }
        ],
    )
    return message.content[0].text

In [ ]:
answer = ask_claude_with_web_context("What is Firecrawl and what can it do?")
print(answer)

### 1.5 Using the Firecrawl Python SDK

The `firecrawl-py` SDK wraps the REST API for convenience.

In [ ]:
from firecrawl import FirecrawlApp

app = FirecrawlApp(api_key=FIRECRAWL_API_KEY)

# Search
search_result = app.search("latest AI news", limit=3)
for r in search_result.get("data", []):
    print(f"- {r.get('title')}: {r.get('url')}")

print()

# Scrape
scrape_result = app.scrape_url("https://firecrawl.dev")
print(f"Scraped: {len(scrape_result.get('markdown', ''))} chars")

### 1.6 Claude Tool Use with Firecrawl

Give Claude direct access to Firecrawl via tool use, so it can decide when to search or scrape.

In [ ]:
import json

tools = [
    {
        "name": "web_search",
        "description": "Search the web for information on a topic. Returns page titles, URLs, and content.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query",
                },
                "limit": {
                    "type": "integer",
                    "description": "Max number of results (default 5)",
                    "default": 5,
                },
            },
            "required": ["query"],
        },
    },
    {
        "name": "scrape_url",
        "description": "Scrape a URL and return its content as clean markdown.",
        "input_schema": {
            "type": "object",
            "properties": {
                "url": {
                    "type": "string",
                    "description": "The URL to scrape",
                }
            },
            "required": ["url"],
        },
    },
]


def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Execute a Firecrawl tool call and return the result."""
    if tool_name == "web_search":
        result = firecrawl_search(tool_input["query"], tool_input.get("limit", 5))
        return json.dumps(result, indent=2)
    elif tool_name == "scrape_url":
        result = firecrawl_scrape(tool_input["url"])
        return json.dumps(result, indent=2)
    return json.dumps({"error": f"Unknown tool: {tool_name}"})


def chat_with_tools(user_message: str) -> str:
    """Chat with Claude, allowing it to use Firecrawl tools as needed."""
    messages = [{"role": "user", "content": user_message}]

    while True:
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        if response.stop_reason == "end_turn":
            return next(
                (block.text for block in response.content if hasattr(block, "text")),
                "",
            )

        # Process tool calls
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = handle_tool_call(block.name, block.input)
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    }
                )

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})


answer = chat_with_tools("What are the main features of Firecrawl? Search the web to find out.")
print(answer)

---
## Part 2: Agent Skills & CLI Integration

Firecrawl provides a CLI and agent skills that give AI coding agents (like Claude Code) direct access to web search, scraping, and interaction.

### 2.1 Install

One command installs the CLI and all agent skills:

```bash
npx -y firecrawl-cli@latest init --all --browser
```

This installs:
- **CLI tools** — `firecrawl search`, `firecrawl scrape`, `firecrawl interact`, and more
- **CLI skills** — `firecrawl/cli`, `firecrawl-search`, `firecrawl-scrape`, `firecrawl-interact`, `firecrawl-crawl`, `firecrawl-map`
- **Build skills** — `firecrawl-build`, `firecrawl-build-onboarding`, `firecrawl-build-scrape`, `firecrawl-build-search`, `firecrawl-build-interact`, `firecrawl-build-crawl`, `firecrawl-build-map`
- **Browser auth** — walks you through sign-in or account creation

If you already have an API key and want to skip browser auth:

```bash
npx -y firecrawl-cli@latest init --all -k fc-YOUR_API_KEY
```

### 2.2 Verify Installation

```bash
firecrawl --status
```

Expected output:
```
  \U0001f525 firecrawl cli v1.14.9

  \u25cf Authenticated via stored credentials
```

### 2.3 CLI Usage Examples

```bash
# Search the web
firecrawl search "latest AI developments" --limit 5

# Search and also scrape full page content
firecrawl search "react hooks tutorial" --scrape --limit 3

# Scrape a single URL
firecrawl scrape "https://docs.firecrawl.dev"

# Save output to a file
firecrawl scrape "https://example.com" -o .firecrawl/example.md

# Map all URLs on a site
firecrawl map "https://docs.firecrawl.dev"

# Crawl an entire docs section
firecrawl crawl "https://docs.firecrawl.dev" --limit 50

# Interact with a page (clicks, forms, navigation)
firecrawl interact "go to amazon.com, search for keyboards, filter by Prime"
```

### 2.4 Workflow Pattern

Follow this escalation pattern for live web work:

1. **Search** — No specific URL yet; find pages and discover sources
2. **Scrape** — Have a URL; extract its content
3. **Map + Scrape** — Large site; find the right URL first, then scrape
4. **Crawl** — Need bulk content from an entire site section
5. **Interact** — Page needs clicks, forms, or login

---
## Part 3: MCP Server Integration

Firecrawl can be added as a [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) server, giving Claude and other LLM clients direct tool access to web scraping and search.

### 3.1 Configuration

Add the following to your MCP client configuration (e.g., `claude_desktop_config.json` for Claude Desktop, or `.claude/settings.json` for Claude Code):

```json
{
  "mcpServers": {
    "firecrawl-mcp": {
      "command": "npx",
      "args": ["-y", "firecrawl-mcp"],
      "env": {
        "FIRECRAWL_API_KEY": "fc-YOUR_API_KEY"
      }
    }
  }
}
```

### 3.2 Setup for Claude Desktop

1. Open Claude Desktop settings
2. Navigate to the MCP servers section
3. Add the configuration above with your API key
4. Restart Claude Desktop

### 3.3 Setup for Claude Code

Add the MCP server to your project or user settings:

```bash
# Via CLI
claude mcp add firecrawl-mcp -- npx -y firecrawl-mcp
```

Or add it directly to `.claude/settings.json`:

```json
{
  "mcpServers": {
    "firecrawl-mcp": {
      "command": "npx",
      "args": ["-y", "firecrawl-mcp"],
      "env": {
        "FIRECRAWL_API_KEY": "fc-YOUR_API_KEY"
      }
    }
  }
}
```

### 3.4 Available MCP Tools

Once configured, the Firecrawl MCP server exposes these tools to the LLM:

| Tool | Description |
|------|-------------|
| `firecrawl_search` | Search the web and return results with content |
| `firecrawl_scrape` | Scrape a URL and return clean markdown |
| `firecrawl_crawl` | Crawl a website starting from a URL |
| `firecrawl_map` | Discover all URLs on a website |
| `firecrawl_extract` | Extract structured data from pages |

### 3.5 MCP vs REST API vs CLI

| Approach | Best for |
|----------|----------|
| **REST API** | Application code, custom integrations, full control |
| **CLI & Skills** | AI coding agents, terminal workflows, live web work |
| **MCP Server** | Claude Desktop, Claude Code, any MCP-compatible client |

---
## API Reference

**Base URL:** `https://api.firecrawl.dev/v1`

**Auth header:** `Authorization: Bearer fc-YOUR_API_KEY`

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/search` | POST | Search the web by query, returns results with optional full content |
| `/scrape` | POST | Extract clean markdown from a single URL |
| `/crawl` | POST | Crawl a website starting from a URL |
| `/map` | POST | Discover all URLs on a website |

Full API documentation: [docs.firecrawl.dev](https://docs.firecrawl.dev)